In [ ]:
import scanpy as sc
import scvi

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_ = adata.copy()
sc.pp.filter_genes(adata_, min_cells=50)
adata_.X = adata_.layers["reads"]
sc.pp.highly_variable_genes(
    adata_,
    n_top_genes=1000,
    # batch_key="batch",
    flavor="seurat_v3",
    inplace=True,
)
adata_ = adata_[:, adata_.var.highly_variable].copy()

In [ ]:
scvi.model.AmortizedLDA.setup_anndata(adata_)
n_topics = 10
model = scvi.model.AmortizedLDA(adata_, n_topics=n_topics)
model.train(
    max_epochs=500,
    batch_size=4096,
    lr=1e-3,
    # early_stopping=True,
    # early_stopping_monitor="elbo_validation",
    # check_val_every_n_epoch=1,
)

In [ ]:
model.history["elbo_train"].plot()

In [ ]:
feature_by_topic = model.get_feature_by_topic()

In [ ]:
topic_prop = model.get_latent_representation()

In [ ]:
adata_.obsm["X_topic"] = topic_prop
sc.pp.neighbors(adata_, use_rep="X_topic")
sc.tl.umap(adata_)

In [ ]:
sc.pl.umap(adata_)

In [ ]:
import numpy as np

# X: (n, d)
C = np.corrcoef(topic_prop, rowvar=False)  # (d, d)

In [ ]:
C

In [ ]:
import seaborn as sns

sns.heatmap(C, cmap="coolwarm")

In [ ]:
for topic in range(n_topics):
    col = f"topic_{topic}"
    genes_to_topic_score = feature_by_topic[col]
    print(col)
    print(genes_to_topic_score.sort_values(ascending=False).head(25))

$$
\begin{align}
\end{align}
$$